# Response Hyperalignment (RHA)

Response Hyperalignment (RHA) aligns subject-specific response patterns into a shared representational space. This section documents the same RHA stages for script-based use and for the graphical interface: searchlight preparation, common-space construction, T matrix calculation, and Alignment.

The examples assume that response data have already been prepared as `.npy` arrays with shape `time points x vertices/voxels` under the HA work directory. Surface folders usually use `hemi-L` and `hemi-R`. CIFTI subcortical folders use `volume-subcortical-L`, `volume-subcortical-R`, and `volume-subcortical-BRAIN_STEM`. NIfTI ROI folders use `volume-<roi_name>`. For details, please refer to the **Data Preparation** section.


## Script-Based Workflow

In a real workflow, the Template, T Matrix, and Alignment steps should <span style="color: red; font-weight: bold;">NOT</span> all be estimated from exactly the same data. The script examples below estimate common spaces and T matrices from `run="01"`, then apply the transformations to `run="02"`. Replace all example paths, subject IDs, task names, ROI names, and file filters with values matching your dataset.

**Required script inputs**

| Item | Meaning |
|---|---|
| `work_dir` | Root HA work directory. It contains subject folders, `logs`, `masks`, common-space outputs, transform outputs, and aligned outputs. |
| `sub_list` | Subject folder names, such as `sub-rid000005`. Use the same subjects for common-space construction, T matrix calculation, and alignment unless your design requires a stage-specific subject set. |


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

import joblib
import nibabel as nib
import neuroboros as nb

from pathlib import Path
from fmriha import gensls, ha

In [ ]:
work_dir = Path("/path/to/ha_workdir")
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]

(rha-searchlight-preparation)=

(searchlight-preparation)=

### (1) Searchlight Preparation

Searchlights define the local neighborhoods used by RHA. 

**Cortical**

For cortical surface data, `neuroboros.sls` is the standard helper used by the examples here. In the example below, it generates vertex-centered searchlights on the ``onavg-ico32`` surface.

Parameters of `nb.sls`:

| Parameter | Default | Meaning |
|---|---|---|
| `lr` | Required | Hemisphere selector, such as `"l"` or `"r"`. |
| `radius` | Required | Surface searchlight radius in millimeters. |
| `space` | `"onavg-ico32"` | Surface space used for searchlight vertices. |
| `center_space` | `None` | Optional surface space used for searchlight centers. With `None`, centers use `space`. |
| `mask` | `True` | Cortical mask for searchlight vertices. It can be a boolean array or a boolean control value. |
| `center_mask` | `None` | Cortical mask for searchlight centers. With `None`, it follows `mask`. |
| `return_dists` | `False` | If `True`, returns `(sls, dists)`. If `False`, returns only `sls`. |


In [ ]:
radius_surf = 20
surface_space = "onavg-ico32"

surface_tmp = {
    lr: nb.sls(
        lr,
        radius_surf,
        surface_space,
        mask=True,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf = {lr: surface_tmp[lr][0] for lr in surface_tmp}
dists_surf = {lr: surface_tmp[lr][1] for lr in surface_tmp}

`gensls.sls_update` is optional. In this example, `mask=False` is used in `nb.sls` so that the searchlights are first generated on the full surface, allowing `gensls.sls_update` to later keep vertices selected by a surface mask and remap the indices.

Parameters of `gensls.sls_update`:

| Parameter | Default | Meaning |
|---|---|---|
| `sls` | Required | Dictionary of searchlight index lists. Keys are usually `"l"` and `"r"`. |
| `dists` | `None` | Dictionary of distance arrays corresponding to `sls`. With `None`, only indices are updated. |
| `surface_mask` | `None` | Dictionary of boolean masks in full-surface indexing. `True` marks vertices retained in the prepared response arrays. |
| `lr` | `None` | Hemispheres to process, for example `"lr"` or `["l", "r"]`. |
| `center_type` | `"first"` | Center-selection rule. `"first"` uses the first vertex in each searchlight. `"idx"` uses the searchlight list position. |


In [ ]:
surface_mask = joblib.load("/path/to/custom_surface_mask.pkl")

full_surface_tmp = {
    lr: nb.sls(
        lr,
        radius_surf,
        surface_space,
        mask=False,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf_full = {lr: full_surface_tmp[lr][0] for lr in full_surface_tmp}
dists_surf_full = {lr: full_surface_tmp[lr][1] for lr in full_surface_tmp}

sls_surf, dists_surf = gensls.sls_update(
    sls_surf_full,
    dists=dists_surf_full,
    surface_mask=surface_mask,
    lr="lr",
    center_type="first",
)

**Subcortical**

For CIFTI subcortical RHA, derive dense left, right, and brain-stem masks from a reference CIFTI file, then generate a volume searchlight inside each mask.

Parameters of `gensls.downsample_cifti_subcortical`:

| Parameter | Default | Meaning |
|---|---|---|
| `cifti_file` | Required | Reference CIFTI file used to read subcortical voxel coordinates and volume dimensions. |
| `N` | `None` | Target number of sampled voxels for downsampled modes. Use `None` with `sls_type="seed"` to keep all available subcortical voxels. |
| `mask3d_out` | `None` | Output directory for saving masks. With `None`, masks are returned in memory. |
| `sls_type` | `"subcortex"` | Sampling mode label. This workflow uses `"seed"` for dense searchlight centers. |
| `voxel_sizes` | `None` | Optional voxel spacing used for spatial sampling. With `None`, voxel-index coordinates are used. |
| `drop_all_zero_columns` | `True` | Removes CIFTI subcortical columns that are zero across all time points before downsampling. |
| `whole_mask` | `False` | When saving masks, controls whether the whole subcortical mask is saved. |
| `hemi_mask` | `True` | When saving masks, controls whether separate left, right, and brain-stem masks are saved. |

Parameters of `gensls.generate_searchlight_vol`:

| Parameter | Default | Meaning |
|---|---|---|
| `mask_dense` | Required | Dense 3D mask defining voxels allowed inside searchlights. It can be a NumPy array or a NIfTI file path. |
| `mask_center` | `None` | Optional 3D mask defining searchlight centers. With `None`, every non-zero voxel in `mask_dense` can be a center. |
| `radius` | `2` | Volume searchlight radius in voxel units. This notebook uses `radius_vol = 4`. |
| `threshold` | `0` | Minimum fraction of in-mask voxels required for a candidate searchlight to be kept. This notebook uses `0.2`. |
| `njobs` | `0` | Number of parallel jobs. `0` runs sequentially. |
| `verbose` | `0` | Joblib verbosity level. |

The returned volume-searchlight dictionary contains `sls_coord`, `sls_idx`, and `dists`. RHA uses `sls_idx` and `dists`.


In [ ]:
reference_cifti = "/path/to/reference_run01.dtseries.nii"
radius_vol = 4

mask_dense_subcortical = {
    lr: gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=None,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

sls_info_subcortical = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_subcortical[lr],
        mask_center=None,
        radius=radius_vol,
        threshold=0.2,
        njobs=10,
        verbose=5,
    )
    for lr in ["l", "r", "brain_stem"]
}

sls_subcortical = {
    f"subcortical-{lr}": sls_info_subcortical[lr]["sls_idx"]
    for lr in ["l", "r", "brain_stem"]
}

dists_subcortical = {
    f"subcortical-{lr}": sls_info_subcortical[lr]["dists"]
    for lr in ["l", "r", "brain_stem"]
}

**Volume ROI**

For an ROI defined by a NIfTI mask, use the non-zero mask voxels as the dense searchlight mask. The ROI name becomes part of the structure folder name, such as `volume-mPFC`. The same `gensls.generate_searchlight_vol` parameter table above applies to this ROI workflow.


In [ ]:
roi_name = "mPFC"
nifti_mask_path = "/path/to/mPFC_mask.nii"

mask_dense_roi = nib.load(nifti_mask_path).get_fdata().astype(bool)

sls_info_roi = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=None,
    radius=radius_vol,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi = {f"volume-{roi_name}": sls_info_roi["sls_idx"]}
dists_roi = {f"volume-{roi_name}": sls_info_roi["dists"]}

(rha-common-space-construction)=

(common-space-construction)=

### (2) Common Space Construction

`ha.cspace_sls` constructs a whole-structure common representational space by computing local searchlight common spaces and aggregating them over the target feature axis.

Parameters of `ha.cspace_sls`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. Input files are searched under `work_dir / sub / step / modality / structure_name`. |
| `sls` | Required | Searchlight indices. Use a list for one structure or a dictionary keyed by hemisphere or volume structure. |
| `dists` | Required | Distance arrays corresponding to `sls`. Required when `weight=True`. |
| `radius` | Required | Searchlight radius used to compute distance weights. Use the same radius used during searchlight preparation. |
| `sub_list` | Required | Subject folder names included in the common-space estimate. |
| `task_name` | `"rha"` | Name written into the common-space output filename as `task-<task_name>`. |
| `cspace_kind` | `"procrustes"` | Common-space algorithm. Supported values include `"procrustes"` and `"pca"`. |
| `common_topography` | `False` | If `True`, maps PCA-like common components back into feature topography. Use `False` for Procrustes RHA. |
| `weight` | `True` | Aggregates overlapping searchlights with distance-based weights. |
| `demean` | `True` | Demeans data before common-template computation. |
| `start_sub` | `0` | Reference subject for Procrustes common-space initialization. Use an integer subject index or `"mean"`. |
| `chunk_size` | `2000` | Number of searchlights processed per chunk. Larger chunks reduce scheduling overhead and require more memory. |
| `njobs` | `5` | Number of joblib workers for the local workflow. |
| `step` | `"resample"` | Prepared-data step folder, usually `"resample"`. |
| `modality` | `"response"` | Data modality folder, usually `"response"` for RHA. |
| `structure_name` | `"hemi-L"` | Structure folder or list of folders to process, such as `"hemi-L"`, `"hemi-R"`, `"volume-subcortical-L"`, or `"volume-mPFC"`. |
| `dtype` | `np.float32` | Numeric dtype for saved common-space arrays. |
| `scope` | `"sls"` | Output naming label for the searchlight scope. If weighted, filenames use `scope-w<scope>`. |
| `save_local` | `True` | If `True`, also saves chunked searchlight-level common spaces for workflows that reuse local common-space targets. |
| `suffix` | `None` | Optional label appended to common-space output filenames as `suffix-<suffix>`. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `dask` | `False` | If `True`, uses Dask-based distributed execution instead of local joblib. |
| `cluster` | `None` | Dask cluster object used when `dask=True`; keep `None` for the standard local workflow. |
| `**file_filters` | Empty | BIDS-style filename filters used to select source `.npy` files, such as `run="01"`, `set="train"`, or `desc="bold-zscore"`. |

Outputs are saved under `work_dir / "common_space" / modality / structure_name`.

**Cortical**

Use the cortical searchlights prepared for `hemi-L` and `hemi-R`.


In [ ]:
file_filters = {"run": "01", "ses": "raiders"}

ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    task_name="rhaPRcortical",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    **file_filters,
)

**Dask version**

To run the workflow with Dask, first create a Dask cluster. The HA inputs remain the same as in the single-machine version; the only changes to the common-space call are setting `dask=True` and passing `cluster=cluster`. The example below uses a SLURM cluster; for detailed Dask jobqueue cluster parameters, see [https://jobqueue.dask.org/en/latest/clusters-api.html](https://jobqueue.dask.org/en/latest/clusters-api.html). Replace the example SLURM parameters below with values appropriate for your computing environment.

In [ ]:
from dask_jobqueue import SLURMCluster

dask_params = {
    "queue": "your_queue",
    "project": "RHA_DASK_DEMO",
    "account": "your_account",
    "cores": 20,
    "memory": "140GB",
    "jobs_num": 4,
    "processes": 1,
    "walltime": "2-00:00:00",
    "scheduler_options": {"dashboard_address": ":1234"},
}

slurm_log_path = Path(work_dir) / "logs" / "slurm_out"
os.makedirs(slurm_log_path, exist_ok=True)

cluster = SLURMCluster(
    queue=dask_params["queue"],
    cores=dask_params["cores"],
    memory=dask_params["memory"],
    processes=dask_params["processes"],
    account=dask_params["account"],
    walltime=dask_params["walltime"],
    scheduler_options=dask_params["scheduler_options"],
    job_name=dask_params["project"],
    log_directory=str(slurm_log_path),
    worker_extra_args=["--death-timeout", "6000"],
)
cluster.scale(jobs=dask_params["jobs_num"])

ha.cspace_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    task_name="rhaPRcortical",
    cspace_kind="procrustes",
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    dask=True,  # bool
    cluster=cluster,  # cluster configuration
    **file_filters,
)

**Subcortical**

Use the CIFTI-derived subcortical searchlights for the left subcortex, right subcortex, and brain stem.


In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="rhaPRsubcortical",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    **file_filters,
)

For the Dask version, the only difference is setting the `dask` parameter to `True` and the `cluster` parameter to the variable `cluster` in the `ha.cspace_sls` call.

**Volume ROI**

Use the ROI searchlights and match `structure_name` to the ROI folder created during data preparation.


In [ ]:
ha.cspace_sls(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    task_name="rhaPRroi",
    cspace_kind="procrustes",
    njobs=32,
    step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    **file_filters,
)

For the Dask version, the only difference is setting the `dask` parameter to `True` and the `cluster` parameter to the variable `cluster` in the `ha.cspace_sls` call.

To construct a PCA template, set `cspace_kind='pca'` and `common_topography=True`. Other parameters, such as `task_name`, can be modified as needed.

(rha-t-matrix-calculation)=

(t-matrix-calculation)=

### (3) T matrix calculation

`ha.xform_sls` calculates subject-specific sparse T matrices by dispatching to the appropriate searchlight transformation workflow. In the examples below, `concatenated=False` uses the local common-space path.

For most analyses, `ha.xform_sls` is the recommended entry point. It keeps the call compact and chooses the matching lower-level workflow from `concatenated` and the available common-space outputs.

Parameters of `ha.xform_sls`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sls` | Required | Searchlight indices matching both the prepared source data and the common-space feature dimension. |
| `dists` | Required | Distance arrays corresponding to `sls`. Required when `xform_weight=True`. |
| `radius` | Required | Searchlight radius used to compute aggregation weights. |
| `sub_list` | Required | Subject folder names for T matrix calculation. |
| `source_step` | `"resample"` | Prepared-data step folder containing the subject data to align. |
| `modality` | `"response"` | Modality folder for source data and transform outputs. |
| `structure_name` | `["hemi-L", "hemi-R"]` | Structure folder or list of folders to process. |
| `task_name` | `"rha1"` | Common-space task name to select and transform task name written to output files. |
| `cspace_kind` | `"procrustes"` | Common-space algorithm used when the un-concatenated workflow needs to locate or compute local common spaces. |
| `cspace_desc` | `None` | Optional filters for selecting the common-space file, such as `{"alg": "procrustes"}`. |
| `concatenated` | `False` | If `True`, uses the saved whole-structure common space through the concatenated workflow. If `False`, uses the un-concatenated/local common-space path. |
| `common_topography` | `False` | Topographic reconstruction setting reused by the un-concatenated fallback workflow. |
| `demean` | `True` | Demeaning setting reused by the un-concatenated fallback workflow. |
| `start_sub` | `0` | Reference subject index reused by the un-concatenated fallback workflow. |
| `reflection` | `True` | Allows reflection in the Procrustes transform. This is the usual RHA setting. |
| `scaling` | `False` | Allows scaling in the Procrustes transform. `False` is typical for RHA. |
| `xform_weight` | `True` | Aggregates local T matrices with distance-based weights. |
| `chunk_size` | `2000` | Number of searchlights processed per chunk. |
| `dtype` | `np.float32` | Numeric dtype used while building and saving sparse T matrices. |
| `njobs` | `5` | Number of joblib workers. |
| `scope` | `"sls"` | Output naming label for the transformation scope. If weighted, filenames use `scope-w<scope>`. |
| `suffix` | `None` | Optional label appended to T matrix filenames as `suffix-<suffix>`. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `verbose` | `1` | Progress verbosity for lower-level workflows. |
| `dask` | `False` | If `True`, uses the current Dask client for T matrix calculation instead of local joblib. |
| `**file_filters` | Empty | BIDS-style filename filters used to select source `.npy` files, such as `run="01"`, `set="train"`, or `desc="bold-zscore"`. |

Outputs are saved under `work_dir / sub / "xforms" / modality / structure_name` as sparse `.npz` matrices.

**Cortical**

Use the same cortical searchlights and task name used for the cortical common space.


In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    task_name="rhaPRcortical",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

**Dask version**

Reuse the cluster/client created in the Template step. Compared with the single-machine version, the T matrix call only sets `dask=True`. If you have not previously run the Template workflow above, create a Dask cluster and connect with `Client(cluster)` first.

In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    task_name="rhaPRcortical",
    cspace_desc={"alg": "procrustes"},
    dask=True,  # bool
    **file_filters,
)

**Subcortical**

Use the subcortical common-space task name and the CIFTI-derived subcortical searchlights.


In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="rhaPRsubcortical",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

For the Dask version, the only difference is to set `dask=True` in the `ha.xform_sls` call.

**Volume ROI**

Use the ROI common-space task name and the ROI searchlights.

In [ ]:
ha.xform_sls(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    task_name="rhaPRroi",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

For the Dask version, the only difference is to set `dask=True` in the `ha.xform_sls` call.

**Additional Transformation Functions**

`ha.xform_sls_con_mega` and `ha.xform_sls_ucon_mega` are available as additional alternatives when chunk-wise sparse accumulation is preferred. For most analyses, `ha.xform_sls` remains the recommended entry point.

| Function | Workflow | Main use |
|---|---|---|
| `ha.xform_sls_con_mega` | Concatenated common-space workflow with chunk-wise sparse accumulation. | Use for large searchlight sets when direct sparse accumulation needs lower peak memory. |
| `ha.xform_sls_ucon_mega` | Local common-space workflow with chunk-wise sparse accumulation. | Use for large local common-space runs with many subjects or searchlights. |

Parameters shared by these two functions:

| Parameter | Meaning |
|---|---|
| `work_dir` | Root HA work directory containing prepared subject data, common spaces, logs, and output folders. |
| `sls` | Searchlight definitions. A list is used for one structure; a dictionary can be used when each structure has its own searchlight list. |
| `dists` | Distance arrays matching `sls`. Required when distance-based transformation weighting is enabled. |
| `radius` | Searchlight radius used together with `dists` to compute aggregation weights. |
| `sub_list` | Subject folder names included in T matrix calculation. |
| `source_step` | Prepared-data step folder containing the source data to align, commonly `"resample"`. |
| `modality` | Modality folder for source data and transformation outputs, commonly `"response"` for RHA. |
| `structure_name` | Structure folder or list of folders to process. |
| `task_name` | Task label used to find the target common space and to name the saved T matrices. |
| `reflection` | Allows reflections in the Procrustes transformation. |
| `scaling` | Allows isotropic scaling in the Procrustes transformation. |
| `chunk_size` | Number of searchlights processed in one batch. Smaller values reduce memory pressure. |
| `dtype` | Numeric dtype used while constructing and saving sparse T matrices. |
| `njobs` | Number of joblib workers. |
| `scope` | Output naming label for the transformation scope. |
| `suffix` | Optional label appended to T matrix filenames as `suffix-<suffix>`. |
| `json_path` | Optional JSON log path. If `None`, defaults to `work_dir / "logs" / "prep_log.json"`. |
| `overwrite` | Controls whether matching JSON records are replaced. |
| `log_num` | Identifier appended to the progress log filename. |
| `**file_filters` | BIDS-style filters used to select source `.npy` files, such as `run="01"`, `set="train"`, or `desc="bold-zscore"`. |

Parameters for `ha.xform_sls_con_mega`:

| Parameter | Meaning |
|---|---|
| `cspace_desc` | Optional BIDS-style filters used to select one common-space file when several files share the same `task_name`. |
| `weight` | Applies distance-based weights while aggregating local transformations. This is the lower-level name of `xform_weight` in `ha.xform_sls`. |

Parameters for `ha.xform_sls_ucon_mega`:

| Parameter | Meaning |
|---|---|
| `cspace_kind` | Algorithm used to estimate local common spaces, such as `"procrustes"`, `"pca"`, `"pcav1"`, `"pcav2"`, or `"cls"`. |
| `common_topography` | Reconstructs local common spaces with a shared topographic pattern when supported by the selected common-space method. |
| `demean` | Demeans the data during local common-space estimation. |
| `start_sub` | Reference subject index used by local common-space estimation. |
| `save_cspace` | Saves the searchlight-level common spaces to disk. |
| `xform_weight` | Applies distance-based weights while aggregating local transformations. |
| `verbose` | Progress verbosity for lower-level workflows. |

The examples below use the same inputs as the `ha.xform_sls` examples above. Select the mega workflow that matches the intended common-space form.

**Cortical**



In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    task_name="rhaPRcortical",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_surf,
    dists=dists_surf,
    radius=radius_surf,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    task_name="rhaPRcortical",
    njobs=32,
    **file_filters,
)


**Subcortical**



In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="rhaPRsubcortical",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_subcortical,
    dists=dists_subcortical,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[
        "volume-subcortical-L",
        "volume-subcortical-R",
        "volume-subcortical-BRAIN_STEM",
    ],
    task_name="rhaPRsubcortical",
    njobs=32,
    **file_filters,
)


**Volume ROI**



In [ ]:
# Concatenated mega workflow: use chunk-wise sparse accumulation for larger runs.
ha.xform_sls_con_mega(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    task_name="rhaPRroi",
    cspace_desc={"alg": "procrustes"},
    njobs=32,
    **file_filters,
)

# Local common-space mega workflow: combine local common spaces with chunk-wise sparse accumulation.
ha.xform_sls_ucon_mega(
    work_dir,
    sls=sls_roi,
    dists=dists_roi,
    radius=radius_vol,
    sub_list=sub_list,
    source_step="resample",
    modality="response",
    structure_name=[f"volume-{roi_name}"],
    task_name="rhaPRroi",
    njobs=32,
    **file_filters,
)

(rha-alignment)=

(alignment)=

### (4) Alignment

`ha.align_pipe` applies each subject's sparse T matrix to selected response data and saves aligned `.npy` arrays under each subject's `align` folder. The aligned matrix is computed as `raw_data @ xform`.

Parameters of `ha.align_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names to align. |
| `source_step` | Required | Step folder containing the response data to align, usually `"resample"` or `"combine"`. |
| `source_modality` | Required | Modality folder containing the response data to align, usually `"response"`. |
| `source_structure_name` | Required | Structure folder containing the response data to align. |
| `source_name_filter` | Required | BIDS-style filters used to select exactly one source `.npy` file per subject, such as `{"run": "02"}`. |
| `xform_modality` | Required | Modality folder under each subject's `xforms` directory, usually `"response"`. |
| `xform_structure_name` | Required | Transform structure folder. Use `None` to use `source_structure_name`. |
| `xform_name_filter` | Required | BIDS-style filters used to select exactly one transform `.npz` file per subject, such as `{"task": "rhaPRcortical"}`. |
| `njobs` | `5` | Number of joblib workers used in the standard local path. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Numeric dtype for saved aligned arrays. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Controls whether matching JSON records are replaced. |
| `log_num` | `"00001"` | Identifier appended to the progress log filename. |
| `dask` | `False` | If `True`, uses the current Dask client for alignment instead of local joblib. |
| `batch_size_dask` | `30` | Number of subjects submitted in each Dask batch. |
| `suffix` | `None` | Optional label appended to the aligned output filename as `suffix-<suffix>`. |

Before alignment, check that each subject has exactly one selected source file and exactly one selected transform file for each structure.

**Cortical**

Apply cortical transformations estimated from `run="01"` to cortical response data from `run="02"`.


In [ ]:
for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter={"run": "02"},
        xform_modality="response",
        xform_structure_name=None,
        xform_name_filter={"task": "rhaPRcortical"},
        njobs=8,
    )

**Dask version**

Reuse the same running cluster/client for alignment. Set `dask=True`; `batch_size_dask` controls how many subjects are submitted per Dask batch. Create a Dask cluster and connect with `Client(cluster)` before running this step. Close the client and cluster after the Dask workflow finishes.

In [ ]:
from dask.distributed import get_client

for structure in ["hemi-L", "hemi-R"]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter={"run": "02"},
        xform_modality="response",
        xform_structure_name=None,
        xform_name_filter={"task": "rhaPRcortical"},
        dask=True,  # bool
        batch_size_dask=30,
    )

client = get_client()
client.close()
cluster.close()

**Subcortical**

Apply subcortical transformations estimated from `run="01"` to subcortical response data from `run="02"`.

In [ ]:
for structure in [
    "volume-subcortical-L",
    "volume-subcortical-R",
    "volume-subcortical-BRAIN_STEM",
]:
    ha.align_pipe(
        work_dir,
        sub_list,
        source_step="resample",
        source_modality="response",
        source_structure_name=structure,
        source_name_filter={"run": "02"},
        xform_modality="response",
        xform_structure_name=None,
        xform_name_filter={"task": "rhaPRsubcortical"},
        njobs=8,
    )

For the Dask version, the only difference is setting the `dask` parameter to `True` and setting `batch_size_dask` as needed in the `ha.align_pipe` call.

**Volume ROI**

Apply ROI transformations estimated from `run="01"` to ROI response data from `run="02"`.

In [ ]:
ha.align_pipe(
    work_dir,
    sub_list,
    source_step="resample",
    source_modality="response",
    source_structure_name=f"volume-{roi_name}",
    source_name_filter={"run": "02"},
    xform_modality="response",
    xform_structure_name=None,
    xform_name_filter={"task": "rhaPRroi"},
    njobs=8,
)

For the Dask version, the only difference is setting the `dask` parameter to `True` and setting `batch_size_dask` as needed in the `ha.align_pipe` call.

## GUI Workflow

Launch the graphical interface with `gui()`. Before configuring CHA stages, fill **Work Dir** at the top of the GUI. 

In [ ]:
from fmriha import gui
gui()

```{image} pic/data_prep/gui.png
:width: 800px
```

Before configuring RHA stages, fill **Work Dir** at the top of the GUI. 

First, click the **n_jobs Parameters** button and specify the number of parallel CPU cores for the **Template Calculation**, **T matrix calculation**, and **Alignment** steps. These values are passed to the `njobs` arguments of `ha.cspace_sls`, `ha.xform_sls`, and `ha.align_pipe`. They have defaults, so editing them is optional unless you want to change parallelism.

```{image} pic/rha/gui_njobs.png
:width: 800px
```

For the GUI, the **Searchlight Configuration** section is shared by RHA and CHA. Click **Space and Searchlight Configuration** first.

For surface data, **Space** and **Surface Searchlight Radius** are required because they are passed to `nb.sls`. **Custom Mask** is optional; if provided, it limits the surface vertices used for preparation and searchlights. Only `.pkl` Surface mask files are supported; retained surface positions must be marked as `True`, and positions to mask out must be marked as `False`. The file should contain a dictionary in the following format:

In [ ]:
{
    'l': left_mask,
    'r': right_mask
}

These settings correspond to the surface part of [Searchlight Preparation](#searchlight-preparation).

```{image} pic/rha/gui_surface.png
:width: 800px
```

For volume data, **Volume Searchlight Radius** and **Threshold** are required when any volume structure is selected; they are passed to `gensls.generate_searchlight_vol` as the local neighborhood radius and minimum in-mask fraction.

For subcortical regions in CIFTI data, **CIFTI Reference** is required. Here, you can use the original CIFTI data from any subject. **NIFTI Mask** and **NIFTI ROI Name** can be left empty for this CIFTI-subcortical case. Parameter meanings are the same as in the [subcortical searchlight script section](#searchlight-preparation).

```{image} pic/rha/gui_cifti_volume.png
:width: 800px
```

For NIFTI data, **NIFTI Mask** and **NIFTI ROI Name** are required. The mask defines the dense volume ROI for searchlight generation, and the ROI name must match the prepared structure folder, such as `volume-mPFC`. **CIFTI Reference** can be left empty for this NIFTI-ROI case.

```{image} pic/rha/gui_nifti.png
:width: 800px
```

After configuring the searchlights, return to the main page and enable the **Template Calculation** section.

The interface in this section is the same as in the script version. Required fields are **Structure**, **Task Name**, **Step**, **Modality**, and **Sub List** when Data Preparation is not being run in the same GUI workflow. **File Filter** rows are required when there is more than one candidate source file; they are optional only if each selected subject/structure folder contains exactly one matching `.npy` file. **Common Space Kind**, **Common Topography**, **Weight**, **Demean**, **Start Sub**, **Chunk Size**, **Float Type**, **Scope**, and **Save Local** have defaults and can be left unchanged if those defaults fit your analysis. **Suffix** is optional and is only needed when you want an extra output label. **File Filter** uses filename entities such as `run`, `set`, `desc`, or `suffix`. For details on each parameter of `ha.cspace_sls`, see the [Common Space Construction section](#common-space-construction).

The following example uses the `hemi-L` and `hemi-R` surface data from a CIFTI dataset to construct a template based on `run-01`.

Note that since Data Preparation has already been run previously, the **Sub List** must be filled in manually here (comma-separated, without quotation marks). If you plan to run Data Preparation and RHA together in a single workflow, this field can be left empty.

```{image} pic/rha/gui_tmpl.png
:width: 800px
```

Next, configure the **T matrix** section in the same way. This panel maps to `ha.xform_sls`. Required fields are **Structure**, **Common Space** (for example `alg=procrustes`), **Task Name**, **Step**, **Modality**, and **Sub List** when Data Preparation is not being run in the same GUI workflow. **File Filter** rows are required when the source files are not uniquely identifiable from the selected folder alone. **Reflection**, **Scaling**, **Weight**, **Concatenated**, **Chunk Size**, **Float Type**, and **Scope** have defaults; change them only when your workflow requires different transform settings. **Suffix** is optional. See the [T matrix calculation section](#t-matrix-calculation) for details on `ha.xform_sls`.

```{image} pic/rha/gui_t.png
:width: 800px
```

Finally, in the **Alignment** section, apply the previously computed T matrices to the `hemi-L` and `hemi-R` data from `run-02`. This panel maps to `ha.align_pipe`. Required fields are **Source Step**, **Source Modality**, **Source Structure**, **Transform Modality**, and **Sub List** when Data Preparation is not being run in the same GUI workflow. **Source Name Filter** and **Transform Name Filter** are required unless the selected source-data folder and transform folder each contain exactly one matching file per subject. **Float Type** has a default, and **Suffix** is optional. **Transform Structure** is shown as text in the GUI and follows the selected source structure; internally this corresponds to passing `xform_structure_name=None`. See the [Alignment section](#alignment) for details on `ha.align_pipe`.

```{image} pic/rha/gui_apply.png
:width: 800px
```

After completing all the settings above, click the **RUN!** button to start the workflow. A configuration file for the current run (with the `.fmriha` extension) will be generated automatically in the **Work Dir**. You can also click **Save Configuration** to save the configuration to another location manually, or use **Load Configuration** to load a previously saved configuration. Note that `.fmriha` files store all settings from the main page, including configurations from subpages such as Data Preparation.

During execution, you can click **Monitor** to check the current job status. 

```{image} pic/rha/gui_monitor.png
:width: 800px
```

If you want to stop the workflow, just click **Terminate All Jobs**.

In the next chapter, we will introduce the configuration of the CHA workflow. CHA is very similar to RHA in terms of workflow design, with the main difference being the addition of a connectivity calculation step.